In [1]:
# CELL 1 - load the simulator straight from Notebook 01 (no copy-paste needed)
import json
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt

nb01 = next(Path.cwd().glob("01_simulation_data*.ipynb"), None)
if nb01 is None:
    raise FileNotFoundError("Put this notebook in the same folder as 01_simulation_data.ipynb")
cells = json.loads(nb01.read_text())["cells"]
sim_src = next("".join(c["source"]) for c in cells if c["cell_type"] == "code"
               and "def build_graph" in "".join(c["source"]) and "simulator ready" in "".join(c["source"]))
exec(compile(sim_src, "notebook01_cell1", "exec"), globals())


simulator ready: True | data folder: /home/ibm-user/floodguard/data/raw


In [4]:
# ================= FloodGuard dashboard (Notebook 06) =================
# Reads your saved outputs only. Nothing in Notebooks 01-05 is changed.
import base64, html as _html, io, json, re, traceback, warnings
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display

ROOT = (Path(RAW).resolve().parent.parent if "RAW" in globals() else Path.cwd().parent)

# ---------------------------------------------------------------- strings (key, English, Kannada)
_STR = [
("app_title","FloodGuard dashboard","ಫ್ಲಡ್‌ಗಾರ್ಡ್ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್"),
("subtitle","Simulated city flood response. All data is simulated.","ಅನುಕರಿತ ನಗರ ಪ್ರವಾಹ ಪ್ರತಿಕ್ರಿಯೆ. ಎಲ್ಲಾ ಡೇಟಾ ಅನುಕರಿತ."),
("menu","Menu","ಮೆನು"), ("lang","Language","ಭಾಷೆ"), ("th_l","Light mode","ಬೆಳಕಿನ ಮೋಡ್"), ("th_d","Dark mode","ಕತ್ತಲೆ ಮೋಡ್"),
("p0","Start here","ಇಲ್ಲಿಂದ ಪ್ರಾರಂಭಿಸಿ"), ("p1","01 · City simulator","01 · ನಗರ ಸಿಮ್ಯುಲೇಟರ್"), ("p2","02 · Data check","02 · ಡೇಟಾ ಪರಿಶೀಲನೆ"),
("p3","03 · Forecast models","03 · ಮುನ್ಸೂಚನೆ ಮಾದರಿಗಳು"), ("p4","04 · What-if actions","04 · ಏನಾದರೆ ಕ್ರಮಗಳು"),
("p5","05 · Decision support","05 · ನಿರ್ಧಾರ ಬೆಂಬಲ"), ("p6","06 · Emergency map","06 · ತುರ್ತು ನಕ್ಷೆ"),
("why","Purpose","ಉದ್ದೇಶ"), ("loading","Loading…","ಲೋಡ್ ಆಗುತ್ತಿದೆ…"),
("err_gen","Something went wrong on this page:","ಈ ಪುಟದಲ್ಲಿ ಏನೋ ತಪ್ಪಾಗಿದೆ:"),
("err_file","A needed file was not found:","ಅಗತ್ಯವಿರುವ ಫೈಲ್ ಕಂಡುಬಂದಿಲ್ಲ:"),
("src","Data sources","ಡೇಟಾ ಮೂಲಗಳು"), ("nf","not found","ಕಂಡುಬಂದಿಲ್ಲ"),
("note_sim","All numbers come from a simulated city. They are results of these runs, not guaranteed outcomes for real floods.",
 "ಎಲ್ಲಾ ಸಂಖ್ಯೆಗಳು ಅನುಕರಿತ ನಗರದಿಂದ ಬಂದವು. ಅವು ಈ ರನ್‌ಗಳ ಫಲಿತಾಂಶಗಳು, ನಿಜವಾದ ಪ್ರವಾಹಗಳ ಖಾತರಿಯ ಫಲಿತಾಂಶಗಳಲ್ಲ."),
("zone","Zone","ವಲಯ"),
("notes","Notes and limits","ಟಿಪ್ಪಣಿಗಳು ಮತ್ತು ಮಿತಿಗಳು"),
("l1","Watch one flood unfold hour by hour","ಒಂದು ಪ್ರವಾಹ ಗಂಟೆಗಂಟೆಗೆ ಹೇಗೆ ಬೆಳೆಯುತ್ತದೆ ನೋಡಿ"),
("l2","Is the data complete and believable?","ಡೇಟಾ ಸಂಪೂರ್ಣ ಮತ್ತು ನಂಬಲರ್ಹವೇ?"),
("l3","How early can a zone be warned?","ಒಂದು ವಲಯಕ್ಕೆ ಎಷ್ಟು ಬೇಗ ಎಚ್ಚರಿಸಬಹುದು?"),
("l4","What helps most: supplies, shelters or roads?","ಯಾವುದು ಹೆಚ್ಚು ಸಹಾಯ: ಸರಬರಾಜು, ಆಶ್ರಯ ಅಥವಾ ರಸ್ತೆಗಳು?"),
("l5","Which zones are at highest risk?","ಯಾವ ವಲಯಗಳು ಹೆಚ್ಚು ಅಪಾಯದಲ್ಲಿವೆ?"),
("l6","Find the shortest and safest route to a hospital","ಆಸ್ಪತ್ರೆಗೆ ಅತಿ ಚಿಕ್ಕ ಮತ್ತು ಸುರಕ್ಷಿತ ಮಾರ್ಗ ಹುಡುಕಿ"), ("sc","Scenario","ಸನ್ನಿವೇಶ"),
# home
("p0_why","A dashboard for the FloodGuard flood-response simulation. Open a page from the ☰ menu.","ಫ್ಲಡ್‌ಗಾರ್ಡ್ ಪ್ರವಾಹ ಪ್ರತಿಕ್ರಿಯೆ ಸಿಮ್ಯುಲೇಶನ್‌ಗಾಗಿ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್. ☰ ಮೆನುವಿನಿಂದ ಒಂದು ಪುಟವನ್ನು ತೆರೆಯಿರಿ."),
("how","How to use it","ಬಳಸುವುದು ಹೇಗೆ"),
("how_t","Use ☰ to switch pages. Use the language and light/dark controls at the top. Hover over a number or label for a short explanation. Start with 01, or go straight to 06 for the map.",
 "ಪುಟಗಳನ್ನು ಬದಲಿಸಲು ☰ ಬಳಸಿ. ಮೇಲಿನ ಭಾಷೆ ಮತ್ತು ಬೆಳಕು/ಕತ್ತಲೆ ನಿಯಂತ್ರಣಗಳನ್ನು ಬಳಸಿ. ಸಣ್ಣ ವಿವರಣೆಗಾಗಿ ಸಂಖ್ಯೆ ಅಥವಾ ಲೇಬಲ್ ಮೇಲೆ ಮೌಸ್ ಇರಿಸಿ. 01 ರಿಂದ ಪ್ರಾರಂಭಿಸಿ, ಅಥವಾ ನಕ್ಷೆಗಾಗಿ ನೇರವಾಗಿ 06 ಕ್ಕೆ ಹೋಗಿ."),
("gl","Plain-language glossary","ಸರಳ ಪದಕೋಶ"),
("g_iso","<b>Cut-off (isolated) zone:</b> a zone with no open road to any shelter.","<b>ಕಡಿತಗೊಂಡ (ಪ್ರತ್ಯೇಕ) ವಲಯ:</b> ಯಾವುದೇ ಆಶ್ರಯಕ್ಕೆ ತೆರೆದ ರಸ್ತೆ ಇಲ್ಲದ ವಲಯ."),
("g_fail","<b>Failed zone:</b> a zone that is cut off, critically short of supplies, or whose shelter is over capacity.","<b>ವಿಫಲ ವಲಯ:</b> ಕಡಿತಗೊಂಡ, ಸರಬರಾಜು ತೀರಾ ಕಡಿಮೆಯಾದ ಅಥವಾ ಆಶ್ರಯ ಸಾಮರ್ಥ್ಯ ಮೀರಿದ ವಲಯ."),
("g_depth","<b>Flood depth:</b> water depth at a zone, in metres.","<b>ಪ್ರವಾಹದ ಆಳ:</b> ವಲಯದಲ್ಲಿ ನೀರಿನ ಆಳ, ಮೀಟರ್‌ಗಳಲ್ಲಿ."),
("g_ph","<b>People-hours failed:</b> people in failed zones, added up over every hour. Bigger means more people affected for longer.","<b>ವಿಫಲ ಜನ-ಗಂಟೆಗಳು:</b> ವಿಫಲ ವಲಯಗಳಲ್ಲಿನ ಜನರನ್ನು ಪ್ರತಿ ಗಂಟೆಗೆ ಕೂಡಿಸಿದ್ದು. ದೊಡ್ಡದಿದ್ದರೆ ಹೆಚ್ಚು ಜನರು ಹೆಚ್ಚು ಕಾಲ ಬಾಧಿತರು."),
("g_shel","<b>Shelter / depot:</b> shelters house displaced people; depots store relief supplies.","<b>ಆಶ್ರಯ / ಡಿಪೋ:</b> ಆಶ್ರಯಗಳು ಸ್ಥಳಾಂತರಗೊಂಡವರಿಗೆ ವಸತಿ ನೀಡುತ್ತವೆ; ಡಿಪೋಗಳು ಪರಿಹಾರ ಸಾಮಗ್ರಿ ಸಂಗ್ರಹಿಸುತ್ತವೆ."),
("g_early","<b>Early warning:</b> a model estimate of whether a zone will fail within 6 hours.","<b>ಮುನ್ನೆಚ್ಚರಿಕೆ:</b> ಒಂದು ವಲಯ 6 ಗಂಟೆಗಳೊಳಗೆ ವಿಫಲವಾಗುವುದೇ ಎಂಬ ಮಾದರಿಯ ಅಂದಾಜು."),
("p0_lim","<b>Not in the project data:</b> hospital locations, hospital capacity, ambulance bases and real road maps. The map page uses hospital zones you choose, and says so.",
 "<b>ಪ್ರಾಜೆಕ್ಟ್ ಡೇಟಾದಲ್ಲಿ ಇಲ್ಲದವು:</b> ಆಸ್ಪತ್ರೆ ಸ್ಥಳಗಳು, ಆಸ್ಪತ್ರೆ ಸಾಮರ್ಥ್ಯ, ಆಂಬ್ಯುಲೆನ್ಸ್ ನೆಲೆಗಳು ಮತ್ತು ನಿಜವಾದ ರಸ್ತೆ ನಕ್ಷೆಗಳು. ನಕ್ಷೆ ಪುಟವು ನೀವು ಆಯ್ಕೆ ಮಾಡುವ ಆಸ್ಪತ್ರೆ ವಲಯಗಳನ್ನು ಬಳಸುತ್ತದೆ ಮತ್ತು ಅದನ್ನು ತಿಳಿಸುತ್ತದೆ."),
# page 1
("p1_why","Shows how rain, flood water, cut-off zones and failures change over 72 hours in one simulated flood.","ಒಂದು ಅನುಕರಿತ ಪ್ರವಾಹದಲ್ಲಿ 72 ಗಂಟೆಗಳಲ್ಲಿ ಮಳೆ, ಪ್ರವಾಹದ ನೀರು, ಕಡಿತಗೊಂಡ ವಲಯಗಳು ಮತ್ತು ವೈಫಲ್ಯಗಳು ಹೇಗೆ ಬದಲಾಗುತ್ತವೆ ಎಂದು ತೋರಿಸುತ್ತದೆ."),
("sc_tip","Each scenario is one simulated flood.","ಪ್ರತಿ ಸನ್ನಿವೇಶವು ಒಂದು ಅನುಕರಿತ ಪ್ರವಾಹ."),
("c_rain","Peak rainfall (mm/h)","ಗರಿಷ್ಠ ಮಳೆ (ಮಿಮೀ/ಗಂ)"), ("c_depth","Peak average flood depth (m)","ಗರಿಷ್ಠ ಸರಾಸರಿ ಪ್ರವಾಹ ಆಳ (ಮೀ)"),
("c_iso","Most zones cut off at once","ಒಂದೇ ಸಮಯದಲ್ಲಿ ಕಡಿತಗೊಂಡ ಗರಿಷ್ಠ ವಲಯಗಳು"), ("c_fail","Zones failed by the last hour","ಕೊನೆಯ ಗಂಟೆಗೆ ವಿಫಲವಾದ ವಲಯಗಳು"),
("c_pop","Population","ಜನಸಂಖ್ಯೆ"),
("ch_rain","Rainfall (mm/h)","ಮಳೆ (ಮಿಮೀ/ಗಂ)"), ("ch_depth","Average flood depth (m)","ಸರಾಸರಿ ಪ್ರವಾಹ ಆಳ (ಮೀ)"),
("ch_iso","Zones cut off from shelters","ಆಶ್ರಯಗಳಿಂದ ಕಡಿತಗೊಂಡ ವಲಯಗಳು"), ("ch_fail","Share of zones failed (0 to 1)","ವಿಫಲ ವಲಯಗಳ ಪಾಲು (0 ರಿಂದ 1)"),
("x_note","Horizontal axis of every chart: hour 0 to 71.","ಎಲ್ಲಾ ಚಾರ್ಟ್‌ಗಳ ಅಡ್ಡ ಅಕ್ಷ: ಗಂಟೆ 0 ರಿಂದ 71."),
("sev_top","Most severe scenarios","ಅತ್ಯಂತ ತೀವ್ರ ಸನ್ನಿವೇಶಗಳು"), ("sev_share","Share of zone-hours failed","ವಿಫಲ ವಲಯ-ಗಂಟೆಗಳ ಪಾಲು"), ("sev_depth","Peak depth (m)","ಗರಿಷ್ಠ ಆಳ (ಮೀ)"),
# page 2
("p2_why","Checks that the saved simulation data is complete and plausible, then shows the Notebook 02 charts.","ಉಳಿಸಿದ ಸಿಮ್ಯುಲೇಶನ್ ಡೇಟಾ ಸಂಪೂರ್ಣ ಮತ್ತು ಸಮಂಜಸವಾಗಿದೆಯೇ ಎಂದು ಪರಿಶೀಲಿಸಿ, ನಂತರ ನೋಟ್‌ಬುಕ್ 02 ಚಾರ್ಟ್‌ಗಳನ್ನು ತೋರಿಸುತ್ತದೆ."),
("q_title","Data quality checks (computed now from the saved file)","ಡೇಟಾ ಗುಣಮಟ್ಟ ಪರಿಶೀಲನೆಗಳು (ಉಳಿಸಿದ ಫೈಲ್‌ನಿಂದ ಈಗ ಲೆಕ್ಕಿಸಿದ್ದು)"),
("q_rows","Row count (expected 763,200)","ಸಾಲುಗಳ ಸಂಖ್ಯೆ (ನಿರೀಕ್ಷೆ 763,200)"), ("q_na","Missing values","ಕಾಣೆಯಾದ ಮೌಲ್ಯಗಳು"),
("q_dup","Duplicate scenario-hour-zone rows","ನಕಲು ಸನ್ನಿವೇಶ-ಗಂಟೆ-ವಲಯ ಸಾಲುಗಳು"), ("q_depth","Flood depth is never negative","ಪ್ರವಾಹದ ಆಳ ಎಂದಿಗೂ ಋಣಾತ್ಮಕವಲ್ಲ"),
("q_road","Open-road share stays between 0 and 1","ತೆರೆದ ರಸ್ತೆ ಪಾಲು 0 ಮತ್ತು 1 ರ ನಡುವೆ ಇದೆ"), ("q_h0","No zone has failed at hour 0","ಗಂಟೆ 0 ರಲ್ಲಿ ಯಾವ ವಲಯವೂ ವಿಫಲವಾಗಿಲ್ಲ"),
("pass","Pass","ಪಾಸ್"), ("chk","Check","ಪರಿಶೀಲಿಸಿ"), ("fig_pick","Saved chart","ಉಳಿಸಿದ ಚಾರ್ಟ್"),
("fig_note","Text inside these saved pictures is in English.","ಈ ಉಳಿಸಿದ ಚಿತ್ರಗಳ ಒಳಗಿನ ಪಠ್ಯ ಇಂಗ್ಲಿಷ್‌ನಲ್ಲಿದೆ."),
("f_scenario_curves","Rain, depth, isolation and failures over time","ಕಾಲದೊಂದಿಗೆ ಮಳೆ, ಆಳ, ಪ್ರತ್ಯೇಕತೆ ಮತ್ತು ವೈಫಲ್ಯಗಳು"),
("f_scenario_severity","How severe the 200 scenarios are","200 ಸನ್ನಿವೇಶಗಳು ಎಷ್ಟು ತೀವ್ರ"),
("f_heatmaps","One scenario: depth, requests, failures by zone and hour","ಒಂದು ಸನ್ನಿವೇಶ: ವಲಯ ಮತ್ತು ಗಂಟೆಯ ಪ್ರಕಾರ ಆಳ, ಕೋರಿಕೆಗಳು, ವೈಫಲ್ಯಗಳು"),
("f_correlation","Which measures move together","ಯಾವ ಅಳತೆಗಳು ಒಟ್ಟಿಗೆ ಚಲಿಸುತ್ತವೆ"),
("f_grid_map","City grid at four moments","ನಾಲ್ಕು ಕ್ಷಣಗಳಲ್ಲಿ ನಗರ ಗ್ರಿಡ್"), ("f_failure_causes","Why zones fail","ವಲಯಗಳು ಏಕೆ ವಿಫಲವಾಗುತ್ತವೆ"),
# page 3
("p3_why","Shows the early-warning models: what they use and the results reported from Notebook 03.","ಮುನ್ನೆಚ್ಚರಿಕೆ ಮಾದರಿಗಳನ್ನು ತೋರಿಸುತ್ತದೆ: ಅವು ಏನು ಬಳಸುತ್ತವೆ ಮತ್ತು ನೋಟ್‌ಬುಕ್ 03 ರಿಂದ ವರದಿ ಮಾಡಿದ ಫಲಿತಾಂಶಗಳು."),
("m_title","Reported model results","ವರದಿ ಮಾಡಿದ ಮಾದರಿ ಫಲಿತಾಂಶಗಳು"),
("m_note","These numbers are typed in from the Notebook 03 output  reported. This dashboard did not recompute them.","ಈ ಸಂಖ್ಯೆಗಳನ್ನು  ವರದಿ ಮಾಡಿದ ನೋಟ್‌ಬುಕ್ 03 ಔಟ್‌ಪುಟ್‌ನಿಂದ ಟೈಪ್ ಮಾಡಲಾಗಿದೆ. ಈ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್ ಅವನ್ನು ಮರುಲೆಕ್ಕಿಸಿಲ್ಲ."),
("m_f1a","Early warning F1 (will a zone fail within 6 hours?)","ಮುನ್ನೆಚ್ಚರಿಕೆ F1 (6 ಗಂಟೆಗಳೊಳಗೆ ವಲಯ ವಿಫಲವಾಗುವುದೇ?)"),
("m_f1b","Flood-driven failure F1","ಪ್ರವಾಹ-ಚಾಲಿತ ವೈಫಲ್ಯ F1"), ("m_ttf","Time-to-failure error (hours, lower is better)","ವೈಫಲ್ಯದವರೆಗಿನ ಸಮಯದ ದೋಷ (ಗಂಟೆಗಳು, ಕಡಿಮೆ ಉತ್ತಮ)"),
("m_model","Model","ಮಾದರಿ"), ("m_base_d","Flood-depth rule","ಪ್ರವಾಹ-ಆಳ ನಿಯಮ"), ("m_base_m","Median guess","ಮಧ್ಯಮ ಊಹೆ"),
("f1_tip","F1 balances catching real failures against false alarms. It runs from 0 to 1; higher is better.","F1 ನಿಜವಾದ ವೈಫಲ್ಯಗಳನ್ನು ಹಿಡಿಯುವುದು ಮತ್ತು ಸುಳ್ಳು ಎಚ್ಚರಿಕೆಗಳ ನಡುವೆ ಸಮತೋಲನ ಮಾಡುತ್ತದೆ. ಇದು 0 ರಿಂದ 1; ಹೆಚ್ಚಿದ್ದರೆ ಉತ್ತಮ."),
("m_files","Saved model files","ಉಳಿಸಿದ ಮಾದರಿ ಫೈಲ್‌ಗಳು"), ("m_feat","Inputs the models use","ಮಾದರಿಗಳು ಬಳಸುವ ಇನ್‌ಪುಟ್‌ಗಳು"),
("m_ok","opened as","ತೆರೆಯಲಾಗಿದೆ:"), ("m_bad","could not be opened:","ತೆರೆಯಲಾಗಲಿಲ್ಲ:"),
("m_split","Training data was split by whole scenario, so no scenario appears in both training and testing.","ತರಬೇತಿ ಡೇಟಾವನ್ನು ಸಂಪೂರ್ಣ ಸನ್ನಿವೇಶದ ಪ್ರಕಾರ ವಿಭಜಿಸಲಾಗಿದೆ, ಆದ್ದರಿಂದ ಯಾವುದೇ ಸನ್ನಿವೇಶ ತರಬೇತಿ ಮತ್ತು ಪರೀಕ್ಷೆ ಎರಡರಲ್ಲೂ ಇಲ್ಲ."),
# page 4
("p4_why","Compares extra supplies, opened shelters and restored roads against doing nothing, on the same floods.","ಅದೇ ಪ್ರವಾಹಗಳಲ್ಲಿ ಹೆಚ್ಚುವರಿ ಸರಬರಾಜು, ತೆರೆದ ಆಶ್ರಯಗಳು ಮತ್ತು ಪುನಃಸ್ಥಾಪಿಸಿದ ರಸ್ತೆಗಳನ್ನು ಏನೂ ಮಾಡದಿರುವುದರೊಂದಿಗೆ ಹೋಲಿಸುತ್ತದೆ."),
("wi_red","Reduction in people-hours failed","ವಿಫಲ ಜನ-ಗಂಟೆಗಳಲ್ಲಿ ಇಳಿಕೆ"), ("wi_redi","Reduction in isolated people-hours","ಕಡಿತಗೊಂಡ ಜನ-ಗಂಟೆಗಳಲ್ಲಿ ಇಳಿಕೆ"),
("wi_n","scenarios in the file","ಫೈಲ್‌ನಲ್ಲಿರುವ ಸನ್ನಿವೇಶಗಳು"), ("wi_act","Action","ಕ್ರಮ"), ("wi_ph","People-hours failed (average)","ವಿಫಲ ಜನ-ಗಂಟೆಗಳು (ಸರಾಸರಿ)"),
("wi_iso","Isolated people-hours (average)","ಕಡಿತಗೊಂಡ ಜನ-ಗಂಟೆಗಳು (ಸರಾಸರಿ)"), ("wi_zf","Zones that failed (average)","ವಿಫಲವಾದ ವಲಯಗಳು (ಸರಾಸರಿ)"),
("wi_note","These results come only from the scenarios saved in the file (the most severe ones), not all 200. They are not guaranteed for other floods. Restored roads can cut isolation a lot while barely changing people-hours failed.",
 "ಈ ಫಲಿತಾಂಶಗಳು ಫೈಲ್‌ನಲ್ಲಿ ಉಳಿಸಿದ ಸನ್ನಿವೇಶಗಳಿಂದ (ಅತ್ಯಂತ ತೀವ್ರವಾದವು) ಮಾತ್ರ ಬಂದಿವೆ, ಎಲ್ಲಾ 200 ಅಲ್ಲ. ಇತರ ಪ್ರವಾಹಗಳಿಗೆ ಖಾತರಿಯಿಲ್ಲ. ರಸ್ತೆ ಪುನಃಸ್ಥಾಪನೆ ಕಡಿತವನ್ನು ಬಹಳ ಕಡಿಮೆ ಮಾಡಬಹುದು, ಆದರೆ ವಿಫಲ ಜನ-ಗಂಟೆಗಳನ್ನು ಅಷ್ಟೇನೂ ಬದಲಿಸದಿರಬಹುದು."),
("s_base","Do nothing","ಏನೂ ಮಾಡದಿರುವುದು"), ("s_sup","Extra supplies","ಹೆಚ್ಚುವರಿ ಸರಬರಾಜು"), ("s_shel","Open extra shelters","ಹೆಚ್ಚುವರಿ ಆಶ್ರಯಗಳನ್ನು ತೆರೆಯುವುದು"),
("s_road","Restore roads","ರಸ್ತೆಗಳ ಪುನಃಸ್ಥಾಪನೆ"), ("s_all","All three combined","ಮೂರೂ ಒಟ್ಟಿಗೆ"),
("s_model","Act where the model predicts failure","ಮಾದರಿ ವೈಫಲ್ಯ ಊಹಿಸುವ ಕಡೆ ಕ್ರಮ"), ("s_rand","Act on random zones","ಯಾದೃಚ್ಛಿಕ ವಲಯಗಳಲ್ಲಿ ಕ್ರಮ"),
("s_oracle","Act with hindsight (oracle)","ಹಿಂದಿನ ತಿಳುವಳಿಕೆಯೊಂದಿಗೆ ಕ್ರಮ (ಒರಾಕಲ್)"),
("pa_title","Does forecasting help choose where to act?","ಎಲ್ಲಿ ಕ್ರಮ ಕೈಗೊಳ್ಳಬೇಕೆಂದು ಆರಿಸಲು ಮುನ್ಸೂಚನೆ ಸಹಾಯ ಮಾಡುತ್ತದೆಯೇ?"),
("pa_same","On people-hours failed, acting on the model's predictions was not clearly better than acting on random zones (difference under 1 percentage point). Compare the other column too.",
 "ವಿಫಲ ಜನ-ಗಂಟೆಗಳಲ್ಲಿ, ಮಾದರಿಯ ಊಹೆಗಳ ಮೇಲೆ ಕ್ರಮ ಕೈಗೊಳ್ಳುವುದು ಯಾದೃಚ್ಛಿಕ ವಲಯಗಳಿಗಿಂತ ಸ್ಪಷ್ಟವಾಗಿ ಉತ್ತಮವಾಗಿರಲಿಲ್ಲ (ವ್ಯತ್ಯಾಸ 1 ಶೇಕಡಾ ಅಂಕಕ್ಕಿಂತ ಕಡಿಮೆ). ಇನ್ನೊಂದು ಕಾಲಂ ಅನ್ನೂ ಹೋಲಿಸಿ."),
("pa_better","On people-hours failed, acting on the model's predictions did better than acting on random zones.","ವಿಫಲ ಜನ-ಗಂಟೆಗಳಲ್ಲಿ, ಮಾದರಿಯ ಊಹೆಗಳ ಮೇಲೆ ಕ್ರಮ ಕೈಗೊಳ್ಳುವುದು ಯಾದೃಚ್ಛಿಕ ವಲಯಗಳಿಗಿಂತ ಉತ್ತಮವಾಗಿತ್ತು."),
("pa_worse","On people-hours failed, acting on the model's predictions did worse than acting on random zones.","ವಿಫಲ ಜನ-ಗಂಟೆಗಳಲ್ಲಿ, ಮಾದರಿಯ ಊಹೆಗಳ ಮೇಲೆ ಕ್ರಮ ಕೈಗೊಳ್ಳುವುದು ಯಾದೃಚ್ಛಿಕ ವಲಯಗಳಿಗಿಂತ ಕೆಟ್ಟದಾಗಿತ್ತು."),
# page 5
("p5_why","Shows the latest zone risk scores saved by Notebook 05 and its situation report, if present.","ನೋಟ್‌ಬುಕ್ 05 ಉಳಿಸಿದ ಇತ್ತೀಚಿನ ವಲಯ ಅಪಾಯ ಅಂಕಗಳು ಮತ್ತು (ಇದ್ದರೆ) ಅದರ ಪರಿಸ್ಥಿತಿ ವರದಿಯನ್ನು ತೋರಿಸುತ್ತದೆ."),
("rf_zones","Zones in the file","ಫೈಲ್‌ನಲ್ಲಿರುವ ವಲಯಗಳು"), ("rf_high","High risk (score 0.5 or more)","ಹೆಚ್ಚಿನ ಅಪಾಯ (ಅಂಕ 0.5 ಅಥವಾ ಹೆಚ್ಚು)"),
("rf_ppl","People in high-risk zones","ಹೆಚ್ಚಿನ ಅಪಾಯದ ವಲಯಗಳಲ್ಲಿನ ಜನರು"), ("rf_iso","Cut off from shelters","ಆಶ್ರಯಗಳಿಂದ ಕಡಿತಗೊಂಡಿವೆ"),
("rf_thr","The 0.5 cut-off is this dashboard's choice for counting, not a value from Notebook 05.","0.5 ಮಿತಿ ಎಣಿಕೆಗಾಗಿ ಈ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್‌ನ ಆಯ್ಕೆ, ನೋಟ್‌ಬುಕ್ 05 ರ ಮೌಲ್ಯವಲ್ಲ."),
("rf_noscn","The scenario and hour are not recorded in this file, so they are not shown.","ಈ ಫೈಲ್‌ನಲ್ಲಿ ಸನ್ನಿವೇಶ ಮತ್ತು ಗಂಟೆ ದಾಖಲಾಗಿಲ್ಲ, ಆದ್ದರಿಂದ ತೋರಿಸಲಾಗಿಲ್ಲ."),
("rf_top","Highest-risk zones","ಅತಿ ಹೆಚ್ಚು ಅಪಾಯದ ವಲಯಗಳು"), ("rf_sum","Saved summary","ಉಳಿಸಿದ ಸಾರಾಂಶ"), ("rf_report","Situation report","ಪರಿಸ್ಥಿತಿ ವರದಿ"),
("rf_src","Report source (read from the saved report)","ವರದಿಯ ಮೂಲ (ಉಳಿಸಿದ ವರದಿಯಿಂದ ಓದಿದ್ದು)"),
("rf_nosrc","The report source is not recorded in the saved report, so it is not shown.","ವರದಿಯ ಮೂಲ ಉಳಿಸಿದ ವರದಿಯಲ್ಲಿ ದಾಖಲಾಗಿಲ್ಲ, ಆದ್ದರಿಂದ ತೋರಿಸಲಾಗಿಲ್ಲ."),
("rf_nofile","Not found (Notebook 05 may not have saved it):","ಕಂಡುಬಂದಿಲ್ಲ (ನೋಟ್‌ಬುಕ್ 05 ಇದನ್ನು ಉಳಿಸಿಲ್ಲದಿರಬಹುದು):"),
("rf_cos","Cloud storage upload status is not tracked by this dashboard, so nothing is claimed about it.","ಕ್ಲೌಡ್ ಸಂಗ್ರಹಣೆ ಅಪ್‌ಲೋಡ್ ಸ್ಥಿತಿಯನ್ನು ಈ ಡ್ಯಾಶ್‌ಬೋರ್ಡ್ ಟ್ರ್ಯಾಕ್ ಮಾಡುವುದಿಲ್ಲ, ಆದ್ದರಿಂದ ಅದರ ಬಗ್ಗೆ ಏನೂ ಹೇಳಲಾಗಿಲ್ಲ."),
("rf_caution","Risk scores are model estimates on simulated data, not certain predictions. Decisions need human judgement.","ಅಪಾಯದ ಅಂಕಗಳು ಅನುಕರಿತ ಡೇಟಾದ ಮೇಲಿನ ಮಾದರಿಯ ಅಂದಾಜುಗಳು, ಖಚಿತ ಮುನ್ಸೂಚನೆಗಳಲ್ಲ. ನಿರ್ಧಾರಗಳಿಗೆ ಮಾನವ ವಿವೇಚನೆ ಬೇಕು."),
("h_node","Zone","ವಲಯ"), ("h_pop","Population","ಜನಸಂಖ್ಯೆ"), ("h_risk","Risk (0 to 1)","ಅಪಾಯ (0 ರಿಂದ 1)"), ("h_eta","Est. hours to failure","ವೈಫಲ್ಯಕ್ಕೆ ಅಂದಾಜು ಗಂಟೆಗಳು"),
("h_depth","Flood depth (m)","ಪ್ರವಾಹದ ಆಳ (ಮೀ)"), ("h_sup","Supply (days)","ಸರಬರಾಜು (ದಿನಗಳು)"), ("h_occ","Shelter fill (0 to 1)","ಆಶ್ರಯ ತುಂಬಿರುವಿಕೆ (0 ರಿಂದ 1)"),
("h_drv","Main driver (rule-based)","ಮುಖ್ಯ ಕಾರಣ (ನಿಯಮ ಆಧಾರಿತ)"),
("d1","cut off from all shelters","ಎಲ್ಲಾ ಆಶ್ರಯಗಳಿಂದ ಕಡಿತಗೊಂಡಿದೆ"), ("d2","supplies running low","ಸರಬರಾಜು ಕಡಿಮೆಯಾಗುತ್ತಿದೆ"),
("d3","nearest shelter near capacity","ಹತ್ತಿರದ ಆಶ್ರಯ ಸಾಮರ್ಥ್ಯದ ಸಮೀಪ"), ("d4","rising flood water","ಏರುತ್ತಿರುವ ಪ್ರವಾಹದ ನೀರು"),
# page 6
("em_why","Shows zones, shelters, depots, hospitals, closed roads and flood depth for one flood and hour, and compares the shortest and the safest route from the patient's zone to a hospital.",
 "ಒಂದು ಪ್ರವಾಹ ಮತ್ತು ಗಂಟೆಗೆ ವಲಯಗಳು, ಆಶ್ರಯಗಳು, ಡಿಪೋಗಳು, ಆಸ್ಪತ್ರೆಗಳು, ಮುಚ್ಚಿದ ರಸ್ತೆಗಳು ಮತ್ತು ಪ್ರವಾಹದ ಆಳವನ್ನು ತೋರಿಸುತ್ತದೆ, ಮತ್ತು ರೋಗಿಯ ವಲಯದಿಂದ ಆಸ್ಪತ್ರೆಗೆ ಅತಿ ಚಿಕ್ಕ ಮತ್ತು ಅತಿ ಸುರಕ್ಷಿತ ಮಾರ್ಗಗಳನ್ನು ಹೋಲಿಸುತ್ತದೆ."),
("em_hour","Hour","ಗಂಟೆ"), ("em_hosp","Hospital zones","ಆಸ್ಪತ್ರೆ ವಲಯಗಳು"), ("em_pat","Patient's zone","ರೋಗಿಯ ವಲಯ"),
("em_hint","Ctrl/Cmd-click to change hospitals. Default: the 3 most populated zones. Patient default: the zone cut off for the most hours.",
 "ಆಸ್ಪತ್ರೆಗಳನ್ನು ಬದಲಿಸಲು Ctrl/Cmd ಒತ್ತಿ ಕ್ಲಿಕ್ ಮಾಡಿ. ಡೀಫಾಲ್ಟ್: ಅತಿ ಹೆಚ್ಚು ಜನಸಂಖ್ಯೆಯ 3 ವಲಯಗಳು. ರೋಗಿ ಡೀಫಾಲ್ಟ್: ಹೆಚ್ಚು ಗಂಟೆ ಕಡಿತಗೊಂಡ ವಲಯ."),
("em_need","The simulator is not loaded. Run the first cell of this notebook, which loads it from Notebook 01, then reopen this page.","ಸಿಮ್ಯುಲೇಟರ್ ಲೋಡ್ ಆಗಿಲ್ಲ. ಈ ನೋಟ್‌ಬುಕ್‌ನ ಮೊದಲ ಸೆಲ್ ಚಲಾಯಿಸಿ (ಅದು ನೋಟ್‌ಬುಕ್ 01 ರಿಂದ ಲೋಡ್ ಮಾಡುತ್ತದೆ), ನಂತರ ಈ ಪುಟವನ್ನು ಮತ್ತೆ ತೆರೆಯಿರಿ."),
("em_len","The road graph has no 'length_km' attribute, so distances cannot be shown. Attributes found:","ರಸ್ತೆ ಗ್ರಾಫ್‌ನಲ್ಲಿ 'length_km' ಗುಣಲಕ್ಷಣ ಇಲ್ಲ, ಆದ್ದರಿಂದ ದೂರಗಳನ್ನು ತೋರಿಸಲಾಗುವುದಿಲ್ಲ. ಕಂಡುಬಂದ ಗುಣಲಕ್ಷಣಗಳು:"),
("em_ok","Replay check passed: flood depth and cut-off zones match the saved data for this scenario.","ಮರುಅನುಕರಣೆ ಪರಿಶೀಲನೆ ಯಶಸ್ವಿ: ಈ ಸನ್ನಿವೇಶಕ್ಕೆ ಪ್ರವಾಹದ ಆಳ ಮತ್ತು ಕಡಿತಗೊಂಡ ವಲಯಗಳು ಉಳಿಸಿದ ಡೇಟಾಗೆ ಹೊಂದುತ್ತವೆ."),
("em_bad","Replay check did NOT match the saved data. Do not rely on this map.","ಮರುಅನುಕರಣೆ ಉಳಿಸಿದ ಡೇಟಾಗೆ ಹೊಂದಲಿಲ್ಲ. ಈ ನಕ್ಷೆಯನ್ನು ನಂಬಬೇಡಿ."),
("em_stat","zones failed","ವಲಯಗಳು ವಿಫಲ"), ("em_cut","cut off from shelters","ಆಶ್ರಯಗಳಿಂದ ಕಡಿತಗೊಂಡಿವೆ"),
("lg_short","Shortest route (fewest km)","ಅತಿ ಚಿಕ್ಕ ಮಾರ್ಗ (ಕಡಿಮೆ ಕಿ.ಮೀ)"), ("lg_safe","Safest route (least distance × water depth)","ಅತಿ ಸುರಕ್ಷಿತ ಮಾರ್ಗ (ಕಡಿಮೆ ದೂರ × ನೀರಿನ ಆಳ)"),
("lg_closed","Road closed by flood","ಪ್ರವಾಹದಿಂದ ಮುಚ್ಚಿದ ರಸ್ತೆ"), ("lg_hosp","Hospital (your setting)","ಆಸ್ಪತ್ರೆ (ನಿಮ್ಮ ಆಯ್ಕೆ)"),
("lg_shel","Shelter","ಆಶ್ರಯ"), ("lg_dep","Depot","ಡಿಪೋ"), ("lg_cut","Zone cut off from shelters (×)","ಆಶ್ರಯಗಳಿಂದ ಕಡಿತಗೊಂಡ ವಲಯ (×)"),
("lg_pat","Circle: patient's zone","ವೃತ್ತ: ರೋಗಿಯ ವಲಯ"), ("lg_fail","Black outline: failed zone","ಕಪ್ಪು ಬಾಹ್ಯರೇಖೆ: ವಿಫಲ ವಲಯ"), ("lg_depth","Blue shade: flood depth (0 to 1 m or more)","ನೀಲಿ ಛಾಯೆ: ಪ್ರವಾಹದ ಆಳ (0 ರಿಂದ 1 ಮೀ ಅಥವಾ ಹೆಚ್ಚು)"),
("rt_route","Route","ಮಾರ್ಗ"), ("rt_hosp","Nearest hospital","ಹತ್ತಿರದ ಆಸ್ಪತ್ರೆ"), ("rt_km","Distance (km)","ದೂರ (ಕಿ.ಮೀ)"),
("rt_fl","Flooded open segments","ಪ್ರವಾಹಕ್ಕೊಳಗಾದ ತೆರೆದ ಭಾಗಗಳು"), ("rt_deep","Deepest (m)","ಅತಿ ಆಳ (ಮೀ)"),
("rt_same","Shortest and safest (same road)","ಅತಿ ಚಿಕ್ಕ ಮತ್ತು ಅತಿ ಸುರಕ್ಷಿತ (ಒಂದೇ ರಸ್ತೆ)"), ("rt_s","Shortest","ಅತಿ ಚಿಕ್ಕ"), ("rt_f","Safest","ಅತಿ ಸುರಕ್ಷಿತ"),
("rt_none","No open road reaches any selected hospital.","ಆಯ್ಕೆ ಮಾಡಿದ ಯಾವುದೇ ಆಸ್ಪತ್ರೆಗೆ ತೆರೆದ ರಸ್ತೆ ಇಲ್ಲ."), ("rt_sel","Select at least one hospital zone.","ಕನಿಷ್ಠ ಒಂದು ಆಸ್ಪತ್ರೆ ವಲಯವನ್ನು ಆಯ್ಕೆಮಾಡಿ."),
("em_assume","Hospital locations are a setting you choose, not project data. The data has no hospital capacity, ambulance bases or real roads, and no travel speed is assumed, so distances are in km only. A segment counts as flooded if it is open but deeper than 0.3 m. This is a decision aid on simulated data, not a dispatch instruction.",
 "ಆಸ್ಪತ್ರೆ ಸ್ಥಳಗಳು ನೀವು ಆಯ್ಕೆ ಮಾಡುವ ಸೆಟ್ಟಿಂಗ್, ಪ್ರಾಜೆಕ್ಟ್ ಡೇಟಾ ಅಲ್ಲ. ಡೇಟಾದಲ್ಲಿ ಆಸ್ಪತ್ರೆ ಸಾಮರ್ಥ್ಯ, ಆಂಬ್ಯುಲೆನ್ಸ್ ನೆಲೆಗಳು ಅಥವಾ ನಿಜವಾದ ರಸ್ತೆಗಳಿಲ್ಲ, ಮತ್ತು ಪ್ರಯಾಣ ವೇಗವನ್ನು ಊಹಿಸಲಾಗಿಲ್ಲ, ಆದ್ದರಿಂದ ದೂರಗಳು ಕಿ.ಮೀ ನಲ್ಲಿ ಮಾತ್ರ. ತೆರೆದಿದ್ದರೂ 0.3 ಮೀ ಗಿಂತ ಆಳವಾದ ಭಾಗವನ್ನು ಪ್ರವಾಹಕ್ಕೊಳಗಾದದ್ದು ಎಂದು ಪರಿಗಣಿಸಲಾಗುತ್ತದೆ. ಇದು ಅನುಕರಿತ ಡೇಟಾದ ನಿರ್ಧಾರ ಸಹಾಯಕ, ರವಾನೆ ಸೂಚನೆಯಲ್ಲ."),
]
STR = {"en": {k: e for k, e, _ in _STR}, "kn": {k: n for k, _, n in _STR}}

THEMES = {
 "light": dict(bg="#ffffff", card="#f3f6f8", ink="#1b2a33", mute="#5b6b75", line="#d3dce2", accent="#2c7fb8", warn="#fff4d6", warn_ink="#6b4a00", err="#fde4e1", err_ink="#8a1c13", ok="#e1f4e8", ok_ink="#14532d"),
 "dark":  dict(bg="#14202a", card="#1d2d39", ink="#e8eef2", mute="#9fb0ba", line="#3a4d5a", accent="#5cb3e6", warn="#4a3b12", warn_ink="#ffe9a8", err="#4d1f1a", err_ink="#ffd0cb", ok="#173d28", ok_ink="#b9efd0")}
STATE = dict(page="p0", lang="en", theme="light")
def tx(k): return STR[STATE["lang"]].get(k) or STR["en"].get(k, k)
def col(): return THEMES[STATE["theme"]]

# ---------------------------------------------------------------- files and data
_found, _cache = {}, {}
def find_file(pattern):
    if pattern not in _found:
        hits = [p for p in ROOT.rglob(pattern) if ".ipynb_checkpoints" not in p.parts]
        _found[pattern] = min(hits, key=lambda p: len(p.parts)) if hits else None
    return _found[pattern]
def load_csv(name):
    p = find_file(name)
    if p is None: raise FileNotFoundError(name)
    return pd.read_csv(p)
def load_sim():
    if "sim" not in _cache:
        p = find_file("sim_runs.parquet") or find_file("sim_runs.csv")
        if p is None: raise FileNotFoundError("sim_runs.parquet / sim_runs.csv")
        _cache["sim"] = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
    return _cache["sim"]

# ---------------------------------------------------------------- html helpers
def esc(s): return _html.escape(str(s))
def ui_box(txt, kind=""):
    c = col(); bg, fg = {"warn": (c["warn"], c["warn_ink"]), "err": (c["err"], c["err_ink"]), "ok": (c["ok"], c["ok_ink"])}.get(kind, (c["card"], c["ink"]))
    return f"<div style='background:{bg};color:{fg};border:1px solid {c['line']};border-radius:10px;padding:10px 14px;margin:8px 0;line-height:1.5'>{txt}</div>"
def ui_cards(items):
    c = col(); out = "".join(f"<div title='{esc(tip)}' style='flex:1 1 150px;background:{c['card']};border:1px solid {c['line']};border-radius:12px;padding:10px 14px'>"
        f"<div style='font-size:.8em;color:{c['mute']}'>{lab}</div><div style='font-size:1.5em;font-weight:700'>{val}</div></div>" for lab, val, tip in items)
    return f"<div style='display:flex;flex-wrap:wrap;gap:10px;margin:8px 0'>{out}</div>"
def ui_table(head, rows):
    c = col(); th = "".join(f"<th style='text-align:left;padding:6px 10px'>{h}</th>" for h in head)
    tr = "".join("<tr>" + "".join(f"<td style='padding:6px 10px;border-top:1px solid {c['line']}'>{x}</td>" for x in r) + "</tr>" for r in rows)
    return f"<div style='overflow-x:auto'><table style='border-collapse:collapse;min-width:480px'><tr>{th}</tr>{tr}</table></div>"
def ui_bar(label, frac, text, color=None, tip=""):
    c = col(); fr = max(0.0, min(1.0, frac)) * 100
    return (f"<div title='{esc(tip)}' style='display:grid;grid-template-columns:minmax(9em,15em) 1fr 5.5em;gap:10px;align-items:center;margin:6px 0'><span>{label}</span>"
            f"<div style='background:{c['line']};border-radius:6px;height:14px'><div style='width:{fr:.1f}%;height:14px;border-radius:6px;background:{color or c['accent']}'></div></div>"
            f"<b style='text-align:right'>{text}</b></div>")
def fig_img(fig, maxw="100%"):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100, facecolor=fig.get_facecolor()); plt.close(fig)
    return f"<img style='width:100%;max-width:{maxw}' src='data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}'>"
def png_img(path): return f"<img style='width:100%;max-width:1100px;background:#fff;border-radius:8px' src='data:image/png;base64,{base64.b64encode(Path(path).read_bytes()).decode()}'>"
def ui_h(t): return f"<h3 style='margin:16px 0 6px'>{t}</h3>"
def purpose(key): return f"<div style='font-size:1.02em;color:{col()['mute']};margin:0 0 10px;line-height:1.5'>{tx(key)}</div>"
def ui_note(txt, title=None):
    c = col(); return (f"<details style='margin:8px 0;font-size:.9em;color:{c['mute']}'><summary style='cursor:pointer'>ⓘ {title or tx('notes')}</summary>"
                       f"<div style='padding:6px 2px;line-height:1.5'>{txt}</div></details>")
def err_html(e):
    tb = traceback.extract_tb(e.__traceback__)[-1]
    return ui_box(f"<b>{tx('err_gen')}</b> {type(e).__name__}: {esc(str(e)[:240])}<br><small>{esc(Path(tb.filename).name)} line {tb.lineno}, in {esc(tb.name)}</small>", "err")
def sources(names):
    rows = []
    for n in names:
        p = find_file(n); rows.append(f"{esc(n)}: " + (f"<code>{esc(p)}</code>" if p else f"<b>{tx('nf')}</b>"))
    return ui_note("<br>".join(rows), tx("src"))
def style_ax(ax):
    c = col(); ax.set_facecolor(c["bg"]); ax.tick_params(colors=c["mute"], labelsize=8)
    for s in ax.spines.values(): s.set_color(c["line"])

# ---------------------------------------------------------------- page 0
def page_p0():
    btns = []
    for n in range(1, 7):
        b = w.Button(description=f"{tx('p'+str(n))}   ·   {tx('l'+str(n))}", layout=w.Layout(width="100%", height="48px"), button_style="info" if n == 6 else "")
        b.on_click(go(f"p{n}")); btns.append(b)
    return [purpose("p0_why"), w.VBox(btns, layout=w.Layout(gap="8px", max_width="760px")), ui_box(tx("p0_lim"), "warn"),
            ui_note("<br>".join(tx(k) for k in ("g_iso", "g_fail", "g_depth", "g_ph", "g_shel", "g_early")), tx("gl")), ui_note(tx("how_t"), tx("how")), ui_note(tx("note_sim"))]

# ---------------------------------------------------------------- page 1
P1 = dict(sc=None)
def page_p1():
    d = load_sim(); sev = d.groupby("scenario")["failed_now"].mean()
    if P1["sc"] is None: P1["sc"] = int(sev.idxmax())
    sl = w.IntSlider(value=P1["sc"], min=int(d["scenario"].min()), max=int(d["scenario"].max()), description=tx("sc"), continuous_update=False,
                     tooltip=tx("sc_tip"), style={"description_width": "90px"}, layout=w.Layout(width="420px"))
    view = w.HTML()
    def draw(*_):
        P1["sc"] = sl.value; g = d[d["scenario"] == sl.value]; by = g.groupby("hour"); last = g[g["hour"] == g["hour"].max()]
        rain, dep, iso, fl = by["rain_mm"].mean(), by["flood_depth"].mean(), by["isolated"].sum(), by["failed_now"].mean()
        cd = ui_cards([(tx("c_rain"), f"{rain.max():.1f}", ""), (tx("c_depth"), f"{dep.max():.2f}", ""), (tx("c_iso"), f"{int(iso.max())}", tx("g_iso").replace("<b>", "").replace("</b>", "")),
                       (tx("c_fail"), f"{int(last['failed_now'].sum())} / {len(last)}", tx("g_fail").replace("<b>", "").replace("</b>", "")), (tx("c_pop"), f"{int(g[g['hour']==0]['population'].sum()):,}", "")])
        imgs = []
        for key, ser, color in (("ch_rain", rain, "#1f77b4"), ("ch_depth", dep, "#17becf"), ("ch_iso", iso, "#d62728"), ("ch_fail", fl, "#9467bd")):
            fig, ax = plt.subplots(figsize=(3.8, 2.4), facecolor=col()["bg"]); style_ax(ax)
            ax.fill_between(ser.index, ser.values, color=color, alpha=.25); ax.plot(ser.index, ser.values, color=color, lw=2); ax.set_xlim(0, 71); fig.tight_layout()
            imgs.append(f"<div style='flex:1 1 280px'><b>{tx(key)}</b>{fig_img(fig)}</div>")
        view.value = cd + f"<div style='display:flex;flex-wrap:wrap;gap:14px'>{''.join(imgs)}</div><div style='font-size:.85em;color:{col()['mute']}'>{tx('x_note')}</div>"
    sl.observe(draw, "value"); draw()
    top = d.groupby("scenario").agg(sh=("failed_now", "mean"), pk=("flood_depth", "max")).sort_values("sh", ascending=False).head(5)
    tb = ui_table([tx("sc"), tx("sev_share"), tx("sev_depth")], [[i, f"{r.sh:.2f}", f"{r.pk:.2f}"] for i, r in top.iterrows()])
    return [purpose("p1_why"), sl, view, ui_h(tx("sev_top")), tb, sources(["sim_runs.parquet", "sim_runs.csv"]), ui_note(tx("note_sim"))]

# ---------------------------------------------------------------- page 2
FIGS = ["scenario_curves", "scenario_severity", "heatmaps", "correlation", "grid_map", "failure_causes"]
def page_p2():
    d = load_sim()
    checks = [("q_rows", len(d) == 763200, f"{len(d):,}"), ("q_na", int(d.isna().sum().sum()) == 0, f"{int(d.isna().sum().sum())}"),
              ("q_dup", int(d.duplicated(['scenario', 'hour', 'node']).sum()) == 0, f"{int(d.duplicated(['scenario', 'hour', 'node']).sum())}"),
              ("q_depth", bool((d['flood_depth'] >= 0).all()), f"min {d['flood_depth'].min():.3f}"),
              ("q_road", bool(d['open_road_frac'].between(0, 1).all()), f"{d['open_road_frac'].min():.2f} to {d['open_road_frac'].max():.2f}"),
              ("q_h0", int(d.loc[d['hour'] == 0, 'failed_now'].sum()) == 0, f"{int(d.loc[d['hour']==0,'failed_now'].sum())}")]
    c = col()
    rows = [[tx(k), det, f"<b style='color:{'#2f8f64' if ok else '#d62728'}'>{tx('pass') if ok else tx('chk')}</b>"] for k, ok, det in checks]
    avail = [f for f in FIGS if find_file(f + ".png")]
    kids = [purpose("p2_why"), ui_h(tx("q_title")), ui_table(["", "", ""], rows)]
    if avail:
        dd = w.Dropdown(options=[(tx("f_" + f), f) for f in avail], description=tx("fig_pick"), layout=w.Layout(width="520px"), style={"description_width": "110px"})
        v = w.HTML(png_img(find_file(avail[0] + ".png")))
        dd.observe(lambda ch: setattr(v, "value", png_img(find_file(ch["new"] + ".png"))), "value")
        kids += [dd, v, f"<div style='font-size:.85em;color:{c['mute']}'>{tx('fig_note')}</div>"]
    return kids + [sources(["sim_runs.parquet", "sim_runs.csv"] + [f + ".png" for f in FIGS]), ui_note(tx("note_sim"))]

# ---------------------------------------------------------------- page 3
REPORTED = [("m_f1a", 0.86, 0.33, "m_base_d", True), ("m_f1b", 0.78, 0.34, "m_base_d", True), ("m_ttf", 3.7, 12.6, "m_base_m", False)]
def page_p3():
    kids = [purpose("p3_why"), ui_h(tx("m_title")), ui_box(tx("m_note"), "warn"), ui_box(tx("f1_tip"))]
    for key, mv, bv, bk, hi in REPORTED:
        vmax = max(mv, bv) * 1.05
        kids.append(f"<div style='margin:12px 0'><b>{tx(key)}</b>" + ui_bar(tx("m_model"), mv / vmax, f"{mv:g}", "#2f8f64") + ui_bar(tx(bk), bv / vmax, f"{bv:g}", "#8a96a0") + "</div>")
    kids += [ui_box(tx("m_split")), ui_h(tx("m_files"))]
    lines = []
    for n in ("failure_clf.joblib", "ttf_reg.joblib", "features.joblib"):
        p = find_file(n)
        if p is None: lines.append(f"{esc(n)}: <b>{tx('nf')}</b>"); continue
        try:
            import joblib
            with warnings.catch_warnings():
                warnings.simplefilter("ignore"); o = joblib.load(p)
            lines.append(f"{esc(n)} ({p.stat().st_size // 1024} KB) {tx('m_ok')} <code>{type(o).__name__}</code>")
            if n == "features.joblib" and isinstance(o, (list, tuple)):
                feats = list(o)
        except Exception as e:
            lines.append(f"{esc(n)} {tx('m_bad')} {type(e).__name__}")
    kids.append(ui_box("<br>".join(lines)))
    if "feats" in locals():
        c = col(); chips = "".join(f"<span style='display:inline-block;margin:3px;padding:3px 9px;border-radius:12px;background:{c['card']};border:1px solid {c['line']};font-size:.85em'>{esc(f)}</span>" for f in feats)
        kids += [ui_h(f"{tx('m_feat')} ({len(feats)})"), chips]
    return kids + [sources(["failure_clf.joblib", "ttf_reg.joblib", "features.joblib"]), ui_note(tx("note_sim"))]

# ---------------------------------------------------------------- page 4
WI = {"baseline (do nothing)": "s_base", "pre-position supplies": "s_sup", "open extra shelters": "s_shel", "restore roads": "s_road", "all three combined": "s_all"}
PA = {"do nothing": "s_base", "model-targeted": "s_model", "random": "s_rand", "oracle (hindsight)": "s_oracle"}
def reduction(df, names, base):
    cols = ["people_hours_failed", "isolated_pop_hours", "zones_ever_failed"]
    g = df.groupby("strategy")[cols].mean()
    if base not in g.index: raise ValueError(f"strategy '{base}' not in file; found {list(g.index)}")
    g = g.reindex([k for k in names if k in g.index]); b = g.loc[base]
    red = pd.DataFrame({c: [(1 - g.loc[k, c] / b[c]) * 100 if b[c] else 0.0 for k in g.index] for c in cols}, index=g.index)
    return g, red
def page_p4():
    wi = load_csv("whatif_results.csv"); g, red = reduction(wi, WI, "baseline (do nothing)")
    bars = "".join(ui_bar(tx(WI[k]), red.loc[k, "people_hours_failed"] / 30, f"{red.loc[k, 'people_hours_failed']:.1f}%") for k in g.index if k != "baseline (do nothing)")
    rows = [[tx(WI[k]), f"{r.people_hours_failed:,.0f}", f"{r.isolated_pop_hours:,.0f}", f"{r.zones_ever_failed:.1f}", f"{red.loc[k,'people_hours_failed']:.1f}%", f"{red.loc[k,'isolated_pop_hours']:.1f}%"] for k, r in g.iterrows()]
    kids = [purpose("p4_why"), ui_h(f"{tx('wi_red')} ({wi['scenario'].nunique()} {tx('wi_n')})"), bars,
            ui_table([tx("wi_act"), tx("wi_ph"), tx("wi_iso"), tx("wi_zf"), tx("wi_red"), tx("wi_redi")], rows), ui_note(tx("wi_note"))]
    pf = find_file("predict_then_act.csv")
    if pf is not None:
        pg, pr = reduction(pd.read_csv(pf), PA, "do nothing")
        pb = "".join(ui_bar(tx(PA[k]), pr.loc[k, "people_hours_failed"] / 30, f"{pr.loc[k, 'people_hours_failed']:.1f}%") for k in pg.index if k != "do nothing")
        prow = [[tx(PA[k]), f"{pr.loc[k,'people_hours_failed']:.1f}%", f"{pr.loc[k,'isolated_pop_hours']:.1f}%"] for k in pg.index if k != "do nothing"]
        note = ""
        if {"model-targeted", "random"} <= set(pg.index):
            diff = pr.loc["model-targeted", "people_hours_failed"] - pr.loc["random", "people_hours_failed"]
            note = ui_box(tx("pa_same" if abs(diff) < 1 else ("pa_better" if diff > 0 else "pa_worse")), "warn")
        kids += [ui_h(tx("pa_title")), pb, ui_table([tx("wi_act"), tx("wi_red"), tx("wi_redi")], prow), note]
    return kids + [sources(["whatif_results.csv", "predict_then_act.csv"]), ui_note(tx("note_sim"))]

# ---------------------------------------------------------------- page 5
def page_p5():
    rp, sp, tp = find_file("risk_forecast.csv"), find_file("forecast_summary*.json"), find_file("situation_report*.md")
    if rp is None and sp is None and tp is None: raise FileNotFoundError("risk_forecast.csv, forecast_summary*.json, situation_report*.md")
    kids = [purpose("p5_why")]
    if rp is not None:
        r = pd.read_csv(rp); hi = r[r["risk_6h"] >= 0.5]
        kids += [ui_cards([(tx("rf_zones"), len(r), ""), (tx("rf_high"), len(hi), tx("rf_thr")), (tx("rf_ppl"), f"{int(hi['population'].sum()):,}", ""), (tx("rf_iso"), int(r["isolated"].sum()), "")]),
                 f"<div style='font-size:.85em'>{tx('rf_thr')} {tx('rf_noscn')}</div>", ui_h(tx("rf_top"))]
        drv = {"cut off from all shelters": "d1", "supplies running low": "d2", "nearest shelter near capacity": "d3", "rising flood water": "d4"}
        kids.append(ui_table([tx(k) for k in ("h_node", "h_pop", "h_risk", "h_eta", "h_depth", "h_sup", "h_occ", "h_drv")],
            [[x.node, f"{int(x.population):,}", f"{x.risk_6h:.2f}", f"{x.est_hours_to_failure:.1f}", f"{x.flood_depth:.2f}", f"{x.days_of_supply:.2f}", f"{x.nearest_shelter_occ:.2f}",
              tx(drv[x.main_driver]) if x.main_driver in drv else esc(x.main_driver)] for x in r.sort_values("risk_6h", ascending=False).head(10).itertuples()]))
    else: kids.append(ui_box(f"{tx('rf_nofile')} risk_forecast.csv", "warn"))
    if sp is not None:
        s = json.loads(sp.read_text()); items = [(k, v) for k, v in s.items() if isinstance(v, (int, float, str))][:10]
        kids += [ui_h(tx("rf_sum")), ui_cards([(esc(k), esc(f"{v:,}" if isinstance(v, int) else v), "") for k, v in items])]
    if tp is not None:
        txt = tp.read_text(); m = re.search(r"_Source:\s*(.+?)_", txt); body = re.sub(r"^#.*\n|_Source:.*_\n", "", txt, flags=re.M).strip()
        kids += [ui_h(tx("rf_report")), ui_box(f"<b>{tx('rf_src')}:</b> {esc(m.group(1))}" if m else tx("rf_nosrc")), ui_box(f"<pre style='white-space:pre-wrap;margin:0;font:inherit'>{esc(body)}</pre>")]
    else: kids.append(ui_box(f"{tx('rf_nofile')} situation_report*.md", "warn"))
    return kids + [ui_note(tx("rf_cos")), ui_note(tx("rf_caution")), sources(["risk_forecast.csv", "forecast_summary*.json", "situation_report*.md"])]

# ---------------------------------------------------------------- page 6: emergency map
def fg_nid(n): return f"{n[0]}_{n[1]}"
def fg_tup(s): x, y = s.split("_"); return int(x), int(y)
def fg_replay(seed, thresh=None):
    th = ROAD_THRESH if thresh is None else thresh
    rng = np.random.default_rng(seed); G = build_graph(rng); rain = make_rainfall(rng)      # same draws as run_simulation
    nodes = list(G.nodes); depth = {n: 0.0 for n in nodes}; trace = []
    for h in range(HOURS):
        new = {}
        for n in nodes:
            nb = [depth[m] for m in G.neighbors(n)]
            inflow = rain[h] / 1000 * (1.0 + max(0, 5.2 - G.nodes[n]["elevation"]))
            spread = 0.15 * (np.mean(nb) - depth[n]) if nb else 0.0
            new[n] = max(0.0, depth[n] + inflow + spread - 0.03 * depth[n] - 0.01)
        depth = new
        trace.append({"depth": dict(depth), "closed": {frozenset((u, v)) for u, v in G.edges if (depth[u] + depth[v]) / 2 > th}})
    return G, trace
def fg_isolated(G, closed):
    H = nx.Graph(); H.add_nodes_from(G.nodes); H.add_edges_from((u, v) for u, v in G.edges if frozenset((u, v)) not in closed)
    reach = set()
    for s in (n for n in G.nodes if G.nodes[n]["ntype"] == "shelter"): reach |= nx.node_connected_component(H, s)
    return {n for n in G.nodes if G.nodes[n]["ntype"] == "zone" and n not in reach}
FLOODED_M = 0.3
def fg_route(G, st, origin, hospitals):
    H = nx.Graph(); H.add_nodes_from(G.nodes)
    for u, v in G.edges:
        if frozenset((u, v)) in st["closed"]: continue
        d = (st["depth"][u] + st["depth"][v]) / 2; km = G.edges[u, v]["length_km"]
        H.add_edge(u, v, km=km, depth=d, expo=km * d + 0.001 * km)
    out = {}
    for name, wt in (("shortest", "km"), ("safest", "expo")):
        dist, paths = nx.single_source_dijkstra(H, origin, weight=wt); cand = [h for h in hospitals if h in dist]
        if not cand: out[name] = None; continue
        hp = min(cand, key=lambda h: dist[h]); p = paths[hp]; legs = [H[a][b] for a, b in zip(p, p[1:])]
        out[name] = dict(path=p, hospital=hp, km=sum(l["km"] for l in legs), flooded=sum(l["depth"] > FLOODED_M for l in legs), deepest=max((l["depth"] for l in legs), default=0.0), segments=len(legs))
    return out
_tr, _ck = {}, {}
def fg_trace(seed):
    if seed not in _tr:
        if len(_tr) > 5: _tr.clear()
        _tr[seed] = fg_replay(seed)
    return _tr[seed]
def fg_check(seed):
    if seed not in _ck:
        G, tr = fg_trace(seed); d = load_sim(); g = d[d["scenario"] == seed]; err, bad = 0.0, 0
        for h, grp in g.groupby("hour"):
            iso = fg_isolated(G, tr[int(h)]["closed"])
            for r in grp.itertuples():
                n = (int(r.x), int(r.y)); err = max(err, abs(float(r.flood_depth) - tr[int(h)]["depth"][n])); bad += int((n in iso) != bool(r.isolated))
        _ck[seed] = bool(err < 1e-4 and bad == 0)
    return _ck[seed]

MS = dict(seed=None, hour=30, hosp=None, pat=None)
def page_p6():
    if not all(n in globals() for n in ("build_graph", "make_rainfall", "HOURS", "ROAD_THRESH")): return [ui_box(tx("em_need"), "err")]
    d = load_sim()
    if MS["seed"] is None: MS["seed"] = int(d.groupby("scenario")["failed_now"].mean().idxmax())
    G0, _ = fg_trace(MS["seed"]); bad = [e for e in G0.edges if "length_km" not in G0.edges[e]]
    if bad: return [ui_box(f"{tx('em_len')} {sorted({k for e in G0.edges for k in G0.edges[e]})}", "err")]
    st_ = {"description_width": "100px"}
    sw = w.IntSlider(value=MS["seed"], min=int(d["scenario"].min()), max=int(d["scenario"].max()), description=tx("sc"), continuous_update=False, tooltip=tx("sc_tip"), style=st_, layout=w.Layout(width="400px"))
    hw = w.IntSlider(value=MS["hour"], min=0, max=HOURS - 1, description=tx("em_hour"), continuous_update=False, style=st_, layout=w.Layout(width="400px"))
    play = w.Play(min=0, max=HOURS - 1, step=1, interval=600); w.jslink((play, "value"), (hw, "value"))
    hosp = w.SelectMultiple(description=tx("em_hosp"), rows=6, layout=w.Layout(width="360px"), style=st_)
    pat = w.Dropdown(description=tx("em_pat"), layout=w.Layout(width="360px"), style=st_)
    info, view = w.HTML(), w.HTML(tx("loading"))

    def draw(seed, h, hosp_ids, pat_id):
        c = col(); G, tr = fg_trace(seed); st = tr[h]; g = d[(d["scenario"] == seed) & (d["hour"] == h)]
        failed = set(g.loc[g["failed_now"] == 1, "node"]); iso = fg_isolated(G, st["closed"]); hs = [fg_tup(s) for s in hosp_ids]; patient = fg_tup(pat_id)
        r = fg_route(G, st, patient, hs) if hs else {"shortest": None, "safest": None}; a, b = r["shortest"], r["safest"]; same = bool(a and b and a["path"] == b["path"])
        fig, ax = plt.subplots(figsize=(6.6, 6.0), facecolor=c["bg"]); ax.set_facecolor(c["bg"])
        for u, v in G.edges:
            cl = frozenset((u, v)) in st["closed"]
            ax.plot([u[0], v[0]], [u[1], v[1]], color="#d62728" if cl else c["line"], lw=2.4 if cl else 1.6, ls="--" if cl else "-", zorder=1)
        if a: ax.plot(*zip(*a["path"]), color="#2f8f64", lw=8, alpha=.85, solid_capstyle="round", zorder=2)
        if b and not same: ax.plot(*zip(*b["path"]), color="#7b4fc9", lw=3.5, ls=(0, (4, 3)), zorder=2.5)
        zn = [n for n in G.nodes if G.nodes[n]["ntype"] == "zone"]
        sc = ax.scatter([n[0] for n in zn], [n[1] for n in zn], c=[st["depth"][n] for n in zn], cmap="Blues", vmin=0, vmax=1, s=[np.sqrt(G.nodes[n]["population"]) * 8 for n in zn],
                        edgecolors=[c["ink"] if fg_nid(n) in failed else "white" for n in zn], linewidths=[3 if fg_nid(n) in failed else .8 for n in zn], zorder=3)
        if iso: ax.scatter([n[0] for n in iso], [n[1] for n in iso], marker="x", color="#d98b1f", s=95, linewidths=2.6, zorder=6)
        for n in G.nodes:
            t = G.nodes[n]["ntype"]
            if t in ("shelter", "depot"):
                ax.scatter(*n, marker="s", s=250, color="#2c5fa8" if t == "shelter" else "#8a96a0", zorder=4); ax.text(*n, "S" if t == "shelter" else "D", color="white", ha="center", va="center", fontweight="bold", zorder=5)
        for hp in hs: ax.scatter(*hp, marker="P", s=430, color="#c8372d", edgecolors="white", linewidths=1.5, zorder=7)
        ax.scatter(*patient, s=900, facecolors="none", edgecolors=c["ink"], linewidths=2.6, zorder=8)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal"); [s.set_visible(False) for s in ax.spines.values()]
        cb = fig.colorbar(sc, ax=ax, shrink=.65); cb.set_label("m", color=c["ink"]); cb.ax.tick_params(colors=c["ink"])   # no Kannada text inside the figure
        img = fig_img(fig, "640px")
        sw_ = lambda colr, lab, dash=False: (f"<span style='display:block;margin:5px 0'><span style='display:inline-block;width:{22 if dash else 12}px;height:{4 if dash else 12}px;background:{colr};margin-right:6px;vertical-align:middle'></span>{lab}</span>")
        legend = (sw_("#2f8f64", tx("lg_short"), True) + sw_("#7b4fc9", tx("lg_safe"), True) + sw_("#d62728", tx("lg_closed"), True) + sw_("#c8372d", tx("lg_hosp")) + sw_("#2c5fa8", tx("lg_shel")) +
                  sw_("#8a96a0", tx("lg_dep")) + sw_("#d98b1f", tx("lg_cut")) + sw_(c["accent"], tx("lg_depth")) + f"<span>{tx('lg_pat')} · {tx('lg_fail')}</span>")
        head = f"<b>{len(failed)} {tx('em_stat')} · {len(iso)} {tx('em_cut')}</b>"
        cells = lambda x: (f"<td colspan='4' style='color:#c8372d'><b>{tx('rt_none')}</b></td>" if x is None else
                           f"<td>{tx('zone')} {fg_nid(x['hospital'])}</td><td>{x['km']:.1f}</td><td>{x['flooded']} / {x['segments']}</td><td>{x['deepest']:.2f}</td>")
        if not hs: tbl = ui_box(tx("rt_sel"), "warn")
        else:
            rows = (f"<tr><td><b>{tx('rt_same')}</b></td>{cells(a)}</tr>" if same else
                    f"<tr><td><b style='color:#2f8f64'>{tx('rt_s')}</b></td>{cells(a)}</tr><tr><td><b style='color:#7b4fc9'>{tx('rt_f')}</b></td>{cells(b)}</tr>")
            tbl = (f"<div style='overflow-x:auto'><table cellpadding='6' style='border-collapse:collapse'><tr style='text-align:left'><th>{tx('rt_route')}</th><th>{tx('rt_hosp')}</th><th>{tx('rt_km')}</th>"
                   f"<th>{tx('rt_fl')}</th><th>{tx('rt_deep')}</th></tr>{rows}</table></div>")
        return (f"<div style='display:flex;flex-wrap:wrap;gap:18px;align-items:flex-start'><div style='flex:1 1 420px;min-width:0'>{img}</div>"
                f"<div style='flex:1 1 300px'>{head}{tbl}<div style='margin-top:10px'>{legend}</div></div></div>")

    busy = [False]
    def refresh(*_):
        if busy[0] or not hosp.options or pat.value is None: return
        MS.update(seed=sw.value, hour=hw.value, hosp=tuple(hosp.value), pat=pat.value)
        try: view.value = draw(sw.value, hw.value, list(hosp.value), pat.value)
        except Exception as e: view.value = err_html(e)
    def on_seed(*_):
        G, _t = fg_trace(sw.value); g = d[d["scenario"] == sw.value]
        zs = sorted([n for n in G.nodes if G.nodes[n]["ntype"] == "zone"], key=lambda n: -G.nodes[n]["population"])
        opts = [(f"{tx('zone')} {fg_nid(n)} ({G.nodes[n]['population']:,})", fg_nid(n)) for n in zs]; ids = [o[1] for o in opts]
        same_sc = MS["hosp"] is not None and MS["seed"] == sw.value and all(x in ids for x in MS["hosp"])
        keep = MS["hosp"] if same_sc else tuple(ids[:3])
        cut = g.groupby("node")["isolated"].sum().sort_values(ascending=False)
        p = MS["pat"] if (same_sc and MS["pat"] in ids) else next((n for n in cut.index if n not in keep and n in ids), ids[min(3, len(ids) - 1)])
        busy[0] = True                                  # options first, then values; no redraw in between
        try:
            hosp.options = opts; hosp.value = keep
            pat.options = opts; pat.value = p
        finally: busy[0] = False
        MS["seed"] = sw.value; ok = fg_check(sw.value)
        info.value = ui_box(tx("em_ok") if ok else tx("em_bad"), "ok" if ok else "err"); refresh()
    def guard(*_):
        try: on_seed()
        except Exception as e: view.value = err_html(e)
    sw.observe(guard, "value")
    for x in (hw, hosp, pat): x.observe(refresh, "value")
    on_seed()
    return [purpose("em_why"), info, w.HBox([sw, hw, play], layout=w.Layout(flex_flow="row wrap")), w.HBox([hosp, pat], layout=w.Layout(flex_flow="row wrap")),
            f"<div style='font-size:.85em;color:{col()['mute']}'>{tx('em_hint')}</div>", view, ui_note(tx("em_assume")), sources(["sim_runs.parquet", "sim_runs.csv"]), ui_note(tx("note_sim"))]

# ---------------------------------------------------------------- shell: menu, theme, language
PAGES = dict(p0=page_p0, p1=page_p1, p2=page_p2, p3=page_p3, p4=page_p4, p5=page_p5, p6=page_p6)
css, title = w.HTML(), w.HTML()
body = w.VBox([])
menu_btn = w.Button(description="☰", layout=w.Layout(width="52px"))
lang_dd = w.Dropdown(options=[("English", "en"), ("ಕನ್ನಡ", "kn")], value="en", layout=w.Layout(width="200px"), style={"description_width": "70px"})
theme_tb = w.ToggleButtons(options=[("☀", "light"), ("☾", "dark")], value="light", style={"button_width": "46px"})
nav_btns = {k: w.Button(layout=w.Layout(width="260px")) for k in PAGES}
nav = w.VBox(list(nav_btns.values()), layout=w.Layout(width="260px", flex="0 0 260px", gap="4px"))

def apply_labels():
    c = col()
    css.value = (f"<style>.fg-root{{background:{c['bg']};color:{c['ink']};padding:12px;border-radius:12px;--jp-widgets-color:{c['ink']};--jp-widgets-label-color:{c['ink']};"
                 f"--jp-ui-font-color1:{c['ink']};--jp-layout-color1:{c['bg']};--jp-widgets-input-color:{c['ink']};--jp-widgets-input-background-color:{c['card']};}}"
                 f".fg-root table td,.fg-root table th,.fg-root h3{{color:{c['ink']}}}.fg-root code{{background:{c['card']};padding:1px 4px;border-radius:4px}}</style>")
    title.value = f"<div style='font-size:1.4em;font-weight:700;color:{c['ink']}'>{tx('app_title')}</div><div style='font-size:.85em;color:{c['mute']}'>{tx('subtitle')}</div>"
    menu_btn.tooltip = tx("menu"); lang_dd.description = tx("lang"); theme_tb.tooltips = [tx("th_l"), tx("th_d")]
    for k, b in nav_btns.items(): b.description = tx(k); b.button_style = "info" if k == STATE["page"] else ""
def show_page():
    head = w.HTML(f"<h2 style='margin:4px 0 2px'>{esc(tx(STATE['page']))}</h2>")
    body.children = [head, w.HTML(ui_box(tx("loading")))]
    try: items = PAGES[STATE["page"]]()
    except FileNotFoundError as e: items = [ui_box(f"<b>{tx('err_file')}</b> {esc(e)}", "err")]
    except Exception as e: items = [err_html(e)]
    body.children = [head] + [x if isinstance(x, w.Widget) else w.HTML(x) for x in items]
def on_menu(_): nav.layout.display = "none" if nav.layout.display != "none" else "flex"   # collapse or show the sidebar
def go(k):
    def f(_): STATE["page"] = k; apply_labels(); show_page()
    return f
def on_lang(ch): STATE["lang"] = ch["new"]; apply_labels(); show_page()
def on_theme(ch): STATE["theme"] = ch["new"]; apply_labels(); show_page()
menu_btn.on_click(on_menu); lang_dd.observe(on_lang, "value"); theme_tb.observe(on_theme, "value")
for k, b in nav_btns.items(): b.on_click(go(k))
header = w.HBox([menu_btn, title, lang_dd, theme_tb], layout=w.Layout(flex_flow="row wrap", align_items="center", gap="12px"))
main = w.VBox([header, body], layout=w.Layout(flex="1 1 480px", min_width="0"))
root = w.VBox([css, w.HBox([nav, main], layout=w.Layout(flex_flow="row wrap", gap="18px", align_items="flex-start"))]); root.add_class("fg-root")
apply_labels(); display(root); show_page()
